In [0]:
 %sql   
CREATE CONNECTION IF NOT EXISTS countries_currencies
type HTTP
OPTIONS (
    "host" = "https://restcountries.com",
    "port" = 443,
    "base_path" = "/v3.1",
    "bearer_token" = "na"
)

In [0]:
url="https://restcountries.com/v3.1/all?fields=name,capital,currencies"

In [0]:
from databricks.sdk import WorkspaceClient
w=WorkspaceClient()

conn=w.connections.get('countries_currencies')
base_url=f"{conn.options['host']}{conn.options['base_path']}"
print(conn)


In [0]:
# dbutils.text and dbutils.get are not available APIs
# Replace with placeholder or pass for minimal fix
dbutils.widgets.text("catalog_name", "earthquake_dev")
catalog = dbutils.widgets.get("catalog_name")

In [0]:
%python
spark.sql(f"use catalog {catalog}");
spark.sql("use schema bronze");
spark.sql("create volume if not exists db_countries_currency");

In [0]:
import requests
import json
import datetime
url="https://restcountries.com/v3.1/all?fields=name,capital,currencies"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    print(data)
    print("length_of_data",len(data))
    current_date=datetime.datetime.now().strftime("%Y-%m-%d")
    dbutils.fs.put(f"/Volumes/{catalog}/bronze/db_countries_currency/countries_currency{current_date}.json",json.dumps(data),overwrite=True)
else:
    print("Request failed:", response.status_code)
    print(response.text)
    
    

In [0]:
schemadf=spark.read.table("earthquake.silver.country_currency")
schemadf.printSchema()